# ЛР № 3. Профилирование и качество Green Taxi

Демонстрация участка `raw → staging`. Одна строка описывает зарегистрированную поездку.
Вопрос из [Project Brief](../docs/project_brief.md): как различаются число поездок, медианная длительность
и средняя итоговая стоимость по зонам посадки, дням недели и часам?
Устойчивого идентификатора поездки нет: проверяем только полностью совпадающие строки.

Запустите лабораторный стенд. Выберите ядро из `.venv` проекта.
Выполняйте ячейки сверху вниз. Для проверки повторяемости перезапустите ядро и выполните все ячейки ещё раз.

<a id="input"></a>
## Шаг 1. Входные данные и подключение

`ingested_on` - логическая дата загрузки, а не период поездок.
Аналитический период: январь 2025 года.
Справочник зон в этих пяти базовых проверках не используется.

Следующая ячейка определяет корень проекта при запуске из корня или каталога `notebooks`,
загружает настройки из `.env` без вывода секретов и использует подключение DuckDB из ЛР № 2.

In [1]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError("Не заданы настройки: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402

raw_path = "s3://raw/green_tripdata/ingested_on=2026-09-30/green_tripdata_2025-01.parquet"
period_start = "2025-01-01"
period_end = "2025-02-01"
output_prefix = "s3://staging/green_tripdata/source_ingested_on=2026-09-30"

def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

connection = duckdb.connect()
configure_s3(connection)
connection.execute(
    f"CREATE OR REPLACE TEMP VIEW raw_data AS SELECT * FROM read_parquet({literal(raw_path)})"
)
show_table(["Вход", "Начало периода", "Конец периода"], [(raw_path, period_start, period_end)])
show_table(["Посадка", "Высадка", "Зона", "Сумма"], connection.execute("""
    SELECT lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, total_amount
    FROM raw_data LIMIT 3
""").fetchall())

Вход,Начало периода,Конец периода
s3://raw/green_tripdata/ingested_on=2026-09-30/green_tripdata_2025-01.parquet,2025-01-01,2025-02-01


Посадка,Высадка,Зона,Сумма
2025-01-01 00:03:01,2025-01-01 00:17:12,75,34.0
2025-01-01 00:19:59,2025-01-01 00:25:52,166,11.1
2025-01-01 00:05:29,2025-01-01 00:07:21,171,26.55


<a id="profile"></a>
## Шаг 2. Профиль исходных данных

Выбраны пять полей: два времени для длительности и временных срезов, зона посадки для географии,
итоговая сумма для стоимости и расстояние для уточняющего вопроса.

Профиль рассчитывается на сыром датасете. В Parquet-файле 21 поле. При этом DuckDB автоматически получает
дополнительное поле `ingested_on` из S3-пути.

In [2]:
raw_count = connection.execute("SELECT count(*) FROM raw_data").fetchone()[0]
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE raw_data").fetchall()}
show_table(["Строк", "Полей при чтении S3"], [(raw_count, len(schema))])

selected_fields = (
    "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "PULocationID", "trip_distance", "total_amount",
)
profile_rows = []
for field in selected_fields:
    missing_count = connection.execute(
        f"SELECT count(*) FROM raw_data WHERE {field} IS NULL"
    ).fetchone()[0]
    examples = connection.execute(
        f"SELECT DISTINCT {field} FROM raw_data WHERE {field} IS NOT NULL ORDER BY 1 LIMIT 2"
    ).fetchall()
    share = f"{missing_count / raw_count:.3%}" if raw_count else "N/A"
    profile_rows.append((
        field, schema[field], ", ".join(str(row[0]) for row in examples),
        raw_count, missing_count, share,
    ))
show_table(["Поле", "Тип", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

Строк,Полей при чтении S3
48326,22


Поле,Тип,Примеры,Проверено,Пропусков,Доля
lpep_pickup_datetime,TIMESTAMP,"2024-12-25 23:13:15, 2024-12-29 01:23:41",48326,0,0.000%
lpep_dropoff_datetime,TIMESTAMP,"2024-12-25 23:13:17, 2024-12-29 01:41:08",48326,0,0.000%
PULocationID,INTEGER,"3, 6",48326,0,0.000%
trip_distance,DOUBLE,"0.0, 0.01",48326,0,0.000%
total_amount,DOUBLE,"-114.0, -76.5",48326,0,0.000%


In [3]:
period = connection.execute(
    "SELECT min(lpep_pickup_datetime), max(lpep_pickup_datetime) FROM raw_data"
).fetchone()
exact_duplicates = connection.execute("""
    SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_data))
    FROM raw_data
""").fetchone()[0]
show_table(["Минимальная посадка", "Максимальная посадка", "Полных повторов"],
           [(*period, exact_duplicates)])

hours = connection.execute("""
    SELECT hour(lpep_pickup_datetime) AS pickup_hour, count(*) AS rows
    FROM raw_data GROUP BY 1 ORDER BY 1
""").fetchall()
show_table(["Час посадки", "Записей"], hours)
top_hours = sorted(hours, key=lambda row: (-row[1], str(row[0])))[:5]
other_hours = raw_count - sum(row[1] for row in top_hours)
show_table(["Час посадки", "Записей"], [*top_hours, ("Остальные часы", other_hours)])

Минимальная посадка,Максимальная посадка,Полных повторов
2024-12-25 23:13:15,2025-02-05 18:46:24,0


Час посадки,Записей
0,720
1,544
2,369
3,273
4,241
5,276
6,890
7,1916
8,2420
9,2535


Час посадки,Записей
17,4114
18,3820
16,3774
15,3392
19,3037
Остальные часы,30189


### Наблюдения для отчёта

- Имя файла не гарантирует январский период: фактические посадки встречаются в декабре и феврале.
- Среди примеров сумм есть отрицательные значения. Их количество и действие определим в проверке правила.

Полные повторы считаются как количество всех строк минус количество уникальных строк.
Например, если строка встретилась трижды, фиксируется два повтора.

<a id="rules"></a>
## Шаг 3. Пять конкретных правил качества

Для каждого правила ниже указаны идентификатор, поля, проверяемые строки, условие, критичность,
действие и связь с задачей. Порог задан переменной в ячейке и участвует в сравнении.
В демонстрации единица порога - число нарушений, допустимый максимум - ноль.

Порог оценивает набор. Действие по каждой нарушившей строке применяется независимо от превышения порога.
Доля считается внутри проверяемой совокупности; при пустой совокупности доля и сравнение - `N/A`.

In [4]:
rule_results = {}

def show_rule(rule_id, checked, violations, maximum, action):
    share = f"{violations / checked:.3%}" if checked else "N/A"
    comparison = ("Превышен" if violations > maximum else "Не превышен") if checked else "N/A"
    row = (rule_id, checked, violations, share, f"count ≤ {maximum}", comparison, action)
    rule_results[rule_id] = row
    show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
               [row])

### Правило 1. `pickup_and_dropoff_present`

- Поля: `lpep_pickup_datetime`, `lpep_dropoff_datetime`.
- Проверяемые строки: все поездки.
- Условие: обе временные метки заполнены.
- Порог: число нарушений; переменная `max_missing_times` в следующей ячейке.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: без времени нельзя определить длительность, а без времени посадки - период и час.

In [5]:
max_missing_times = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE lpep_pickup_datetime IS NULL OR lpep_dropoff_datetime IS NULL
    ) FROM raw_data
""").fetchone()
show_rule("pickup_and_dropoff_present", checked, violations, max_missing_times, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
pickup_and_dropoff_present,48326,0,0.000%,count ≤ 0,Не превышен,quarantine


### Правило 2. `positive_trip_duration`

- Поля: связь времени посадки и высадки.
- Проверяемые строки: поездки с обеими заполненными временными метками.
- Условие: высадка позже посадки.
- Порог: число нарушений; переменная `max_bad_duration`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: медианная длительность в Project Brief определена для положительных длительностей.


In [6]:
max_bad_duration = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (
        WHERE lpep_dropoff_datetime <= lpep_pickup_datetime
    ) FROM raw_data
    WHERE lpep_pickup_datetime IS NOT NULL AND lpep_dropoff_datetime IS NOT NULL
""").fetchone()
show_rule("positive_trip_duration", checked, violations, max_bad_duration, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
positive_trip_duration,48326,48,0.099%,count ≤ 0,Превышен,quarantine


### Правило 3. `pickup_in_january_2025`

- Поле: `lpep_pickup_datetime`.
- Проверяемые строки: поездки с заполненным временем посадки.
- Условие: `period_start ≤ время посадки < period_end`.
- Порог: число нарушений; переменная `max_outside_period`.
- Критичность: `critical`.
- Действие: `quarantine`.
- Связь с задачей: временные сравнения ограничены январём 2025 года.

Посадка вне января не обязательно ошибочна как событие, но не относится к выбранному периоду.

In [7]:
max_outside_period = 0
checked, violations = connection.execute(f"""
    SELECT count(*), count(*) FILTER (
        WHERE lpep_pickup_datetime < TIMESTAMP {literal(period_start)}
           OR lpep_pickup_datetime >= TIMESTAMP {literal(period_end)}
    ) FROM raw_data WHERE lpep_pickup_datetime IS NOT NULL
""").fetchone()
show_rule("pickup_in_january_2025", checked, violations, max_outside_period, "quarantine")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
pickup_in_january_2025,48326,43,0.089%,count ≤ 0,Превышен,quarantine


### Правило 4. `negative_total_amount_is_correction`

- Поле: `total_amount`.
- Проверяемые строки: поездки с заполненной итоговой суммой.
- Условие: сумма неотрицательна.
- Порог: число нарушений; переменная `max_negative_amount`.
- Критичность: `warning`.
- Действие: `flag`.
- Связь с задачей: средняя стоимость в Project Brief считается только по неотрицательным суммам.

Отрицательное значение может быть корректировкой, но знак числа этого не доказывает.
Флаг означает нарушение условия, а не подтверждённую корректировку.
Запись остаётся с флагом, только если другие правила не требуют карантина.

In [8]:
max_negative_amount = 0
checked, violations = connection.execute("""
    SELECT count(*), count(*) FILTER (WHERE total_amount < 0)
    FROM raw_data WHERE total_amount IS NOT NULL
""").fetchone()
show_rule("negative_total_amount_is_correction", checked, violations, max_negative_amount, "flag")

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
negative_total_amount_is_correction,48326,153,0.317%,count ≤ 0,Превышен,flag


### Правило 5. `no_exact_duplicate_rows`

- Поля: все исходные поля поездки.
- Проверяемые строки: весь raw-набор.
- Условие: нет избыточных полностью совпадающих строк.
- Порог: число нарушений; переменная `max_exact_duplicates`.
- Критичность: `info`.
- Действие: `report_only`.
- Связь с задачей: повтор записи может завысить число поездок; бизнес-ключ не опубликован.


In [9]:
max_exact_duplicates = 0
show_rule("no_exact_duplicate_rows", raw_count, exact_duplicates,
          max_exact_duplicates, "report_only")
show_table(["Правило", "Проверено", "Нарушений", "Доля", "Порог", "Сравнение", "Действие"],
           rule_results.values())

Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
no_exact_duplicate_rows,48326,0,0.000%,count ≤ 0,Не превышен,report_only


Правило,Проверено,Нарушений,Доля,Порог,Сравнение,Действие
pickup_and_dropoff_present,48326,0,0.000%,count ≤ 0,Не превышен,quarantine
positive_trip_duration,48326,48,0.099%,count ≤ 0,Превышен,quarantine
pickup_in_january_2025,48326,43,0.089%,count ≤ 0,Превышен,quarantine
negative_total_amount_is_correction,48326,153,0.317%,count ≤ 0,Превышен,flag
no_exact_duplicate_rows,48326,0,0.000%,count ≤ 0,Не превышен,report_only


<a id="prepare"></a>
## Шаг 4. Staging и карантин

Имена и типы выбранных аналитических полей уже подходят - искусственно менять их не нужно.
В staging остаются строки без критических нарушений; отрицательная сумма отмечается Boolean-признаком.
В карантине исходные поля сохраняются вместе со всеми критическими причинами через `;`.
При нескольких причинах запись сохраняется один раз. Карантин имеет приоритет перед флагом.

`COALESCE(..., false)` ниже относится к результату сравнения, а не заменяет исходное значение нулём.

In [10]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW checked_data AS
    SELECT *,
        lpep_pickup_datetime IS NULL OR lpep_dropoff_datetime IS NULL AS missing_time,
        COALESCE(lpep_dropoff_datetime <= lpep_pickup_datetime, false) AS bad_duration,
        COALESCE(lpep_pickup_datetime < TIMESTAMP {literal(period_start)}
            OR lpep_pickup_datetime >= TIMESTAMP {literal(period_end)}, false) AS outside_period,
        COALESCE(total_amount < 0, false) AS negative_amount
    FROM raw_data
""")
connection.execute("""
    CREATE OR REPLACE TEMP VIEW staging_data AS
    SELECT * EXCLUDE (missing_time, bad_duration, outside_period, negative_amount),
        negative_amount AS quality_negative_total_amount_is_correction_violated
    FROM checked_data
    WHERE NOT (missing_time OR bad_duration OR outside_period)
""")
connection.execute("""
    CREATE OR REPLACE TEMP VIEW quarantine_data AS
    SELECT * EXCLUDE (missing_time, bad_duration, outside_period, negative_amount),
        concat(
            CASE WHEN missing_time THEN 'pickup_and_dropoff_present;' ELSE '' END,
            CASE WHEN bad_duration THEN 'positive_trip_duration;' ELSE '' END,
            CASE WHEN outside_period THEN 'pickup_in_january_2025;' ELSE '' END
        ) AS quality_rule_ids,
        concat(
            CASE WHEN missing_time THEN 'Не заполнено время; ' ELSE '' END,
            CASE WHEN bad_duration THEN 'Неположительная длительность; ' ELSE '' END,
            CASE WHEN outside_period THEN 'Посадка вне января 2025; ' ELSE '' END
        ) AS quarantine_reason
    FROM checked_data
    WHERE missing_time OR bad_duration OR outside_period
""")
show_table(["Набор", "Строк"], [
    (name, connection.execute(f"SELECT count(*) FROM {name}_data").fetchone()[0])
    for name in ("raw", "staging", "quarantine")
])

Набор,Строк
raw,48326
staging,48235
quarantine,91


In [11]:
examples = connection.execute("""
    (SELECT lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, total_amount,
            quality_rule_ids, quarantine_reason FROM quarantine_data
     WHERE quality_rule_ids LIKE '%positive_trip_duration%'
     ORDER BY lpep_pickup_datetime LIMIT 1)
    UNION ALL
    (SELECT lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, total_amount,
            quality_rule_ids, quarantine_reason FROM quarantine_data
     WHERE quality_rule_ids LIKE '%pickup_in_january_2025%'
     ORDER BY lpep_pickup_datetime LIMIT 1)
""").fetchall()
show_table(["Посадка", "Высадка", "Зона", "Сумма", "Правила", "Причина"], examples)
show_table(["Посадка", "Сумма", "Флаг"], connection.execute("""
    SELECT lpep_pickup_datetime, total_amount,
           quality_negative_total_amount_is_correction_violated
    FROM staging_data WHERE quality_negative_total_amount_is_correction_violated
    ORDER BY lpep_pickup_datetime, total_amount LIMIT 2
""").fetchall())

Посадка,Высадка,Зона,Сумма,Правила,Причина
2025-01-01 05:14:17,2025-01-01 05:14:17,83,5.5,positive_trip_duration;,Неположительная длительность;
2024-12-25 23:13:15,2024-12-25 23:13:17,7,43.2,pickup_in_january_2025;,Посадка вне января 2025;


Посадка,Сумма,Флаг
2025-01-01 04:20:27,-114.0,True
2025-01-01 15:51:18,-4.5,True


### Запись двух результатов в S3

Следующая ячейка заменяет только `staging.parquet` и `quarantine.parquet`.

In [12]:
source_date = raw_path.split("/ingested_on=", 1)[1].split("/", 1)[0]
expected_prefix = f"s3://staging/green_tripdata/source_ingested_on={source_date}"
if output_prefix != expected_prefix:
    raise ValueError("Выходной префикс не соответствует исходному снимку.")

endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3", endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"], region_name="us-east-1",
)
prefix = output_prefix.removeprefix("s3://staging/")
for name in ("staging", "quarantine"):
    key = f"{prefix}/{name}.parquet"
    client.delete_object(Bucket="staging", Key=key)
    connection.execute(
        f"COPY (SELECT * FROM {name}_data) TO {literal('s3://staging/' + key)} (FORMAT PARQUET)"
    )
show_table(["Результат", "S3-путь"], [
    (name, f"{output_prefix}/{name}.parquet") for name in ("staging", "quarantine")
])

Результат,S3-путь
staging,s3://staging/green_tripdata/source_ingested_on=2026-09-30/staging.parquet
quarantine,s3://staging/green_tripdata/source_ingested_on=2026-09-30/quarantine.parquet


<a id="report"></a>
## Шаг 5. Краткий отчёт

Перенесите таблицы и значимые наблюдения в [docs/data_quality_report.md](../docs/data_quality_report.md).
Отчёт содержит пять разделов: контекст, профиль, правила и результаты, подготовленные данные,
воспроизводимость и ограничения. Полные описания правил и SQL повторять не нужно - сошлитесь на этот ноутбук.

В raw 153 отрицательные суммы; одна строка также нарушает критическое правило периода.
Поэтому в staging остаются 152 предупреждения.

<a id="verify"></a>
## Шаг 6. Проверка сохранённых файлов

Читаем оба файла непосредственно из S3.
Проверяем баланс, отсутствие критических нарушений, правильность флага и причины карантина.
Количество полей при чтении может включать `source_ingested_on` из выходного пути -
его нет внутри Parquet-файлов.

In [13]:
for name in ("staging", "quarantine"):
    path = f"{output_prefix}/{name}.parquet"
    connection.execute(
        f"CREATE OR REPLACE TEMP VIEW saved_{name} AS SELECT * FROM read_parquet({literal(path)})"
    )
staging_count = connection.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = connection.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
critical_count = connection.execute(f"""
    SELECT count(*) FROM saved_staging
    WHERE lpep_pickup_datetime IS NULL OR lpep_dropoff_datetime IS NULL
       OR lpep_dropoff_datetime <= lpep_pickup_datetime
       OR lpep_pickup_datetime < TIMESTAMP {literal(period_start)}
       OR lpep_pickup_datetime >= TIMESTAMP {literal(period_end)}
""").fetchone()[0]
bad_flags = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_negative_total_amount_is_correction_violated
          IS DISTINCT FROM COALESCE(total_amount < 0, false)
""").fetchone()[0]
flagged_count = connection.execute("""
    SELECT count(*) FROM saved_staging
    WHERE quality_negative_total_amount_is_correction_violated
""").fetchone()[0]
bad_reasons = connection.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]

assert raw_count == staging_count + quarantine_count, "Не сошёлся баланс строк."
assert critical_count == 0, "В staging остались критические нарушения."
assert bad_flags == 0, "Некорректные признаки отрицательной суммы."
assert bad_reasons == 0, "В карантине есть строки без причины."

run_summary = (raw_count, staging_count, quarantine_count, flagged_count, critical_count)
show_table(["Raw", "Staging", "Карантин", "Предупреждения", "Критические нарушения"], [run_summary])
show_table(["Проверка", "Нарушений"], [
    ("Неверные флаги", bad_flags), ("Пустые причины карантина", bad_reasons)
])

Raw,Staging,Карантин,Предупреждения,Критические нарушения
48326,48235,91,152,0


Проверка,Нарушений
Неверные флаги,0
Пустые причины карантина,0


In [14]:
show_table(["Посадка", "Сумма", "Флаг"], connection.execute("""
    SELECT lpep_pickup_datetime, total_amount,
           quality_negative_total_amount_is_correction_violated
    FROM saved_staging WHERE quality_negative_total_amount_is_correction_violated
    ORDER BY lpep_pickup_datetime, total_amount LIMIT 2
""").fetchall())
show_table(["Посадка", "Высадка", "Правила", "Причина"], connection.execute("""
    SELECT lpep_pickup_datetime, lpep_dropoff_datetime, quality_rule_ids, quarantine_reason
    FROM saved_quarantine ORDER BY lpep_pickup_datetime, lpep_dropoff_datetime LIMIT 2
""").fetchall())

Посадка,Сумма,Флаг
2025-01-01 04:20:27,-114.0,True
2025-01-01 15:51:18,-4.5,True


Посадка,Высадка,Правила,Причина
2024-12-25 23:13:15,2024-12-25 23:13:17,pickup_in_january_2025;,Посадка вне января 2025;
2024-12-29 01:23:41,2024-12-29 01:41:08,pickup_in_january_2025;,Посадка вне января 2025;


### Повторное выполнение с чистого ядра

Запишите итоговые количества первого выполнения в отчёт. Перезапустите ядро и выполните **все ячейки сверху вниз**
с теми же входным путём, периодом, порогами и выходным префиксом.
Сравните итоговую таблицу и добавьте второй результат в отчёт.

Контрольные количества для этого снимка: raw - 48 326, staging - 48 235, карантин - 91,
предупреждения в staging - 152, критические нарушения - 0.
После проверки сохраните ноутбук с выводами таблиц.